# AOPC Benchmark

Ноутбук для простого сравнения `IG`, `NAA` и нескольких конфигураций `Cheap-IG` по метрике `AOPC` на `100` изображениях из `Oxford Pets`.

Здесь нет `NAOPC`-нормализации, `candidate_top_k` и `beam-search`: benchmark ранжирует все несхлопнутые нейроны выбранного слоя и меряет изменение score на фиксированной budget-сетке perturbation.

Текущая конфигурация notebook сравнивает три positive-only варианта `Cheap-IG` на сегменте `[0, 0.2]` с `top_k = 8000 / 16000 / 32000` и hybrid-fill `naa_scaled`. Для второго прогона достаточно поменять `CHEAP_IG_FILL_RHO` с `0.8` на `1.0`.

## Импорты

In [1]:
from pathlib import Path

from IPython.display import Markdown, display

from modules.aopc_benchmark import benchmark_classifier_aopc, classifier_method_spec


## Параметры

Первый запуск на uncached `100` изображениях может быть долгим. Повторные запуски должны опираться на `core-cache`, `method-cache` и `evaluation-cache`.

In [2]:
OXFORD_PETS_DIR = Path("oxford_pets")
N_IMAGES = 100

CLASSIFIER_LAYER = "model.6"
UNIT_MODE = "neuron"  # alternatives: "spatial_cell", "filter"
N_STEPS = 128
PERTURBATION_MODE = "deletion"  # alternatives: "deletion", "insertion"
BUDGET_MODE = "percent_steps"  # alternative: "fractions"
BUDGET_STEP_PERCENT = 1.0
BUDGET_NUM_STEPS = 100
CLEAR_EVERY = 8
FD_EPS = 1e-3

CHEAP_IG_SEGMENT_START = 0.0
CHEAP_IG_SEGMENT_END = 0.2
CHEAP_IG_SELECTION_MODE = "positive"
CHEAP_IG_SELECTION_TOP_K_VALUES = [8000, 16000, 32000]
CHEAP_IG_FILL_MODE = "naa_scaled"  # alternatives: "zero", "naa_scaled"
CHEAP_IG_FILL_RHO = 1  # rerun with 1.0 for the second launch

CACHE_ROOT = Path("output/aopc_cache")
OUTPUT_DIR = Path(
    f"output/aopc_classifier_{UNIT_MODE}_{PERTURBATION_MODE}_{BUDGET_MODE}_oxford_pets_100_"
    f"cheapig_pos_seg02_{CHEAP_IG_FILL_MODE}_rho{CHEAP_IG_FILL_RHO:g}"
)
REFRESH_CORE = False
REFRESH_METHODS = False
REFRESH_EVALUATIONS = False


In [3]:
def collect_oxford_pets_images(image_dir=OXFORD_PETS_DIR, n_images=N_IMAGES):
    image_dir = Path(image_dir)
    if not image_dir.exists():
        raise FileNotFoundError(f"Oxford Pets directory not found: {image_dir}")

    image_paths = []
    for pattern in ("*.jpg", "*.jpeg", "*.png", "*.webp"):
        image_paths.extend(image_dir.glob(pattern))

    image_paths = sorted(set(image_paths), key=lambda path: path.name.lower())
    if len(image_paths) < n_images:
        raise ValueError(
            f"Requested {n_images} images, but found only {len(image_paths)} in {image_dir}"
        )
    return [str(path) for path in image_paths[:n_images]]


IMAGE_PATHS = collect_oxford_pets_images()
len(IMAGE_PATHS), IMAGE_PATHS[:5]


(100,
 ['oxford_pets/Abyssinian_1.jpg',
  'oxford_pets/Abyssinian_108.jpg',
  'oxford_pets/Abyssinian_117.jpg',
  'oxford_pets/Abyssinian_126.jpg',
  'oxford_pets/Abyssinian_135.jpg'])

## Методы

In [4]:
cheap_ig_variants = [
    classifier_method_spec(
        "cheap_ig",
        name=(
            f"Cheap-IG+[0,0.2]/k{top_k}/"
            f"{CHEAP_IG_FILL_MODE}/rho{CHEAP_IG_FILL_RHO:g}"
        ),
        segment_start=CHEAP_IG_SEGMENT_START,
        segment_end=CHEAP_IG_SEGMENT_END,
        selection_mode=CHEAP_IG_SELECTION_MODE,
        selection_top_k=top_k,
        fill_mode=CHEAP_IG_FILL_MODE,
        fill_rho=CHEAP_IG_FILL_RHO,
    )
    for top_k in CHEAP_IG_SELECTION_TOP_K_VALUES
]

METHOD_SPECS = [
    classifier_method_spec("ig", name="IG"),
    *cheap_ig_variants,
    classifier_method_spec("naa", name="NAA"),
]

METHOD_SPECS


[{'kind': 'ig', 'name': 'IG'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 8000,
  'fill_mode': 'naa_scaled',
  'fill_rho': 1,
  'name': 'Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 16000,
  'fill_mode': 'naa_scaled',
  'fill_rho': 1,
  'name': 'Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1'},
 {'kind': 'cheap_ig',
  'segment_start': 0.0,
  'segment_end': 0.2,
  'selection_mode': 'positive',
  'selection_top_k': 32000,
  'fill_mode': 'naa_scaled',
  'fill_rho': 1,
  'name': 'Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1'},
 {'kind': 'naa', 'name': 'NAA'}]

## Запуск Бенчмарка

In [5]:
results = benchmark_classifier_aopc(
    image_paths=IMAGE_PATHS,
    method_specs=METHOD_SPECS,
    layer_name=CLASSIFIER_LAYER,
    unit_mode=UNIT_MODE,
    n_steps=N_STEPS,
    perturbation_mode=PERTURBATION_MODE,
    budget_mode=BUDGET_MODE,
    budget_step_fraction=BUDGET_STEP_PERCENT / 100.0,
    budget_num_steps=BUDGET_NUM_STEPS,
    clear_every=CLEAR_EVERY,
    fd_eps=FD_EPS,
    cache_root=CACHE_ROOT,
    target_dir=OUTPUT_DIR,
    save_output=True,
    refresh_core=REFRESH_CORE,
    refresh_methods=REFRESH_METHODS,
    refresh_evaluations=REFRESH_EVALUATIONS,
)

print("output_dir:", results["output_dir"])
print("report_path:", results["report_path"])
print("summary_path:", results["summary_path"])


output_dir: /Users/ashentide/PycharmProjects/PaperImplementations/output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1
report_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/aopc_report.md
summary_path: /Users/ashentide/PycharmProjects/PaperImplementations/output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/aopc_summary.json


## Markdown-Отчёт

In [ ]:
display(Markdown(results["report_markdown"]))


# AOPC Benchmark

- task=`classifier`
- layer_name=`model.6`
- n_steps=128
- unit_mode=`neuron`
- perturbation_mode=`deletion`
- budget_mode=`percent_steps`
- budget_step_fraction=0.0100
- budget_num_steps=100
- effective_budget_steps=100
- n_images=100
- cache_root=`output/aopc_cache`

## Aggregate Summary

| Method | Deletion AOPC / clean_delta | Mean Rank | Attr Runtime (s) | Eval Runtime (s) | Abs Error |
| --- | ---: | ---: | ---: | ---: | ---: |
| IG | 1.1031 +- 0.1743 | 2.6100 +- 1.4205 | 6.5987 +- 0.6687 | 1.6433 +- 0.5753 | 0.7237 +- 0.6076 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 1.0543 +- 0.0807 | 2.7300 +- 0.5071 | 4.6292 +- 2.2582 | 1.6696 +- 0.6026 | 95.3462 +- 20.9398 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 1.0366 +- 0.0731 | 3.7900 +- 0.5531 | 4.5855 +- 2.1831 | 1.6391 +- 0.5775 | 97.3697 +- 21.1971 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 1.0215 +- 0.0640 | 4.5400 +- 0.9531 | 4.6029 +- 2.2029 | 1.6183 +- 0.5370 | 97.9022 +- 21.2363 |
| NAA | 1.1774 +- 0.1549 | 1.3300 +- 0.7079 | 3.4544 +- 0.6663 | 1.6207 +- 0.5078 | 13.1379 +- 3.1544 |

## Core Summary

| Metric | Value |
| --- | ---: |
| clean_delta | 14.0641 +- 2.9601 |
| n_units_total | 50176.0000 +- 0.0000 |
| n_budget_steps | 100.0000 +- 0.0000 |
| core_runtime_s | 0.4682 +- 0.1197 |

## Figures

### aopc_summary

![](output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/figures/aopc_summary.png)

### aopc_distributions

![](output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/figures/aopc_distributions.png)

### aopc_curves

![](output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/figures/aopc_curves.png)

### aopc_pairwise_wins

![](output/aopc_classifier_neuron_deletion_percent_steps_oxford_pets_100_cheapig_pos_seg02_naa_scaled_rho1/figures/aopc_pairwise_wins.png)

## Per-Image Score

| Image | IG | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | NAA |
| --- | ---: | ---: | ---: | ---: | ---: |
| Abyssinian_1.jpg | 1.4202 | 1.2276 | 1.1844 | 1.1422 | 1.4980 |
| Abyssinian_108.jpg | 0.6774 | 0.9107 | 0.8890 | 0.8736 | 0.9575 |
| Abyssinian_117.jpg | 0.9479 | 0.9666 | 0.9473 | 0.9494 | 0.9658 |
| Abyssinian_126.jpg | 1.3837 | 1.1476 | 1.1169 | 1.0831 | 1.3827 |
| Abyssinian_135.jpg | 0.7639 | 0.8798 | 0.8747 | 0.8795 | 0.9011 |
| Abyssinian_144.jpg | 0.7602 | 0.9473 | 0.9444 | 0.9503 | 0.9163 |
| Abyssinian_154.jpg | 0.8904 | 0.9223 | 0.9147 | 0.9166 | 1.0046 |
| Abyssinian_165.jpg | 1.3293 | 1.2119 | 1.1618 | 1.1196 | 1.4364 |
| Abyssinian_175.jpg | 1.3548 | 1.1437 | 1.1167 | 1.0865 | 1.4012 |
| Abyssinian_184.jpg | 1.3050 | 1.1216 | 1.0964 | 1.0677 | 1.3085 |
| Abyssinian_2.jpg | 0.8398 | 0.9626 | 0.9589 | 0.9595 | 1.0265 |
| Abyssinian_212.jpg | 0.9698 | 1.0308 | 1.0077 | 0.9793 | 1.1386 |
| Abyssinian_224.jpg | 0.7312 | 0.8752 | 0.8796 | 0.8899 | 0.8595 |
| Abyssinian_29.jpg | 0.8476 | 0.9646 | 0.9502 | 0.9489 | 1.0419 |
| Abyssinian_43.jpg | 1.3156 | 1.0809 | 1.0338 | 0.9884 | 1.3082 |
| Abyssinian_52.jpg | 0.9716 | 0.9542 | 0.9427 | 0.9363 | 1.0137 |
| Abyssinian_63.jpg | 1.0297 | 1.0089 | 1.0084 | 1.0028 | 1.0247 |
| Abyssinian_73.jpg | 1.0940 | 1.0784 | 1.0559 | 1.0373 | 1.2141 |
| Abyssinian_83.jpg | 1.5279 | 1.2840 | 1.2309 | 1.1746 | 1.6181 |
| Abyssinian_92.jpg | 0.8823 | 1.0098 | 0.9843 | 0.9639 | 1.2243 |
| american_bulldog_108.jpg | 0.9448 | 1.0510 | 1.0399 | 1.0302 | 1.1767 |
| american_bulldog_117.jpg | 1.1616 | 1.1176 | 1.0811 | 1.0463 | 1.3409 |
| american_bulldog_126.jpg | 0.8422 | 1.0126 | 1.0056 | 1.0040 | 1.0693 |
| american_bulldog_135.jpg | 1.2402 | 1.1250 | 1.1006 | 1.0812 | 1.2894 |
| american_bulldog_144.jpg | 1.2077 | 1.0674 | 1.0455 | 1.0241 | 1.2776 |
| american_bulldog_158.jpg | 1.1885 | 1.1105 | 1.0774 | 1.0457 | 1.2269 |
| american_bulldog_173.jpg | 1.1380 | 1.0442 | 1.0252 | 1.0058 | 1.1623 |
| american_bulldog_182.jpg | 0.9456 | 0.9462 | 0.9416 | 0.9462 | 0.9938 |
| american_bulldog_191.jpg | 1.1978 | 1.1178 | 1.0995 | 1.0810 | 1.2286 |
| american_bulldog_200.jpg | 1.1830 | 1.0340 | 1.0198 | 1.0098 | 1.1402 |
| american_bulldog_212.jpg | 1.1090 | 1.0255 | 1.0061 | 0.9921 | 1.1962 |
| american_bulldog_24.jpg | 1.0445 | 0.9540 | 0.9437 | 0.9370 | 0.9781 |
| american_bulldog_33.jpg | 1.2304 | 1.1589 | 1.1219 | 1.0891 | 1.3651 |
| american_bulldog_43.jpg | 1.1275 | 1.0910 | 1.0721 | 1.0527 | 1.1774 |
| american_bulldog_52.jpg | 0.9100 | 1.0412 | 1.0297 | 1.0243 | 1.0659 |
| american_bulldog_61.jpg | 0.9687 | 0.9701 | 0.9612 | 0.9582 | 0.9686 |
| american_bulldog_70.jpg | 1.1825 | 1.0939 | 1.0623 | 1.0337 | 1.2548 |
| american_bulldog_8.jpg | 1.0611 | 1.0328 | 1.0225 | 1.0094 | 1.0948 |
| american_bulldog_9.jpg | 1.1105 | 1.0321 | 1.0174 | 1.0053 | 1.1342 |
| american_bulldog_99.jpg | 0.9642 | 0.9914 | 0.9827 | 0.9805 | 1.0449 |
| american_pit_bull_terrier_107.jpg | 0.9095 | 0.9835 | 0.9734 | 0.9643 | 1.1242 |
| american_pit_bull_terrier_116.jpg | 1.1409 | 1.0037 | 0.9819 | 0.9646 | 1.1847 |
| american_pit_bull_terrier_125.jpg | 1.0932 | 1.0300 | 1.0215 | 1.0131 | 1.0999 |
| american_pit_bull_terrier_134.jpg | 1.1174 | 1.0487 | 1.0307 | 1.0157 | 1.1583 |
| american_pit_bull_terrier_143.jpg | 1.0406 | 1.0013 | 0.9885 | 0.9779 | 1.0712 |
| american_pit_bull_terrier_152.jpg | 1.1442 | 1.1766 | 1.1530 | 1.1244 | 1.3882 |
| american_pit_bull_terrier_161.jpg | 1.0075 | 1.0551 | 1.0541 | 1.0515 | 1.1173 |
| american_pit_bull_terrier_170.jpg | 1.1112 | 1.0140 | 1.0011 | 0.9895 | 1.1175 |
| american_pit_bull_terrier_18.jpg | 1.5072 | 1.2655 | 1.2066 | 1.1497 | 1.6636 |
| american_pit_bull_terrier_189.jpg | 0.9998 | 0.9749 | 0.9698 | 0.9701 | 0.9731 |
| american_pit_bull_terrier_198.jpg | 0.9157 | 0.9670 | 0.9546 | 0.9527 | 1.0072 |
| american_pit_bull_terrier_22.jpg | 1.1232 | 1.1054 | 1.0889 | 1.0679 | 1.1857 |
| american_pit_bull_terrier_32.jpg | 1.2580 | 1.1250 | 1.1030 | 1.0784 | 1.2742 |
| american_pit_bull_terrier_42.jpg | 1.0508 | 1.0167 | 1.0109 | 1.0094 | 1.1303 |
| american_pit_bull_terrier_51.jpg | 1.1475 | 1.0688 | 1.0511 | 1.0318 | 1.1530 |
| american_pit_bull_terrier_60.jpg | 1.2365 | 1.1174 | 1.0966 | 1.0762 | 1.2672 |
| american_pit_bull_terrier_7.jpg | 1.1238 | 1.0313 | 1.0210 | 1.0115 | 1.1582 |
| american_pit_bull_terrier_79.jpg | 0.9780 | 0.9466 | 0.9396 | 0.9360 | 1.0128 |
| american_pit_bull_terrier_9.jpg | 1.0220 | 0.9836 | 0.9822 | 0.9836 | 0.9925 |
| american_pit_bull_terrier_99.jpg | 1.0467 | 1.0454 | 1.0242 | 1.0097 | 1.2162 |
| basset_hound_107.jpg | 1.0257 | 1.0373 | 1.0292 | 1.0247 | 1.1387 |
| basset_hound_116.jpg | 1.2852 | 1.1626 | 1.1322 | 1.1019 | 1.3719 |
| basset_hound_125.jpg | 1.0914 | 1.0467 | 1.0263 | 1.0100 | 1.1654 |
| basset_hound_134.jpg | 0.9636 | 1.0333 | 1.0404 | 1.0464 | 1.0312 |
| basset_hound_143.jpg | 1.1156 | 1.0051 | 0.9999 | 0.9958 | 1.0332 |
| basset_hound_152.jpg | 1.1345 | 1.0415 | 1.0306 | 1.0219 | 1.1633 |
| basset_hound_161.jpg | 1.2238 | 1.0916 | 1.0686 | 1.0458 | 1.2342 |
| basset_hound_170.jpg | 1.2246 | 1.0900 | 1.0673 | 1.0517 | 1.2291 |
| basset_hound_18.jpg | 1.0251 | 1.0008 | 0.9906 | 0.9896 | 1.1011 |
| basset_hound_189.jpg | 1.0960 | 1.0562 | 1.0387 | 1.0251 | 1.1433 |
| basset_hound_198.jpg | 1.0274 | 0.9985 | 0.9856 | 0.9743 | 1.1606 |
| basset_hound_26.jpg | 1.1375 | 1.0390 | 1.0322 | 1.0248 | 1.1362 |
| basset_hound_35.jpg | 1.2397 | 1.1751 | 1.1624 | 1.1455 | 1.2827 |
| basset_hound_44.jpg | 1.0498 | 1.1013 | 1.0887 | 1.0747 | 1.2408 |
| basset_hound_53.jpg | 1.0352 | 1.0042 | 0.9972 | 0.9882 | 1.0822 |
| basset_hound_62.jpg | 1.2173 | 1.1026 | 1.0905 | 1.0763 | 1.2675 |
| basset_hound_71.jpg | 1.3066 | 1.1957 | 1.1747 | 1.1494 | 1.3979 |
| basset_hound_80.jpg | 0.9324 | 1.0372 | 1.0332 | 1.0321 | 1.1228 |
| basset_hound_9.jpg | 1.2199 | 1.0799 | 0.9980 | 0.9519 | 1.3096 |
| basset_hound_99.jpg | 1.1971 | 1.0964 | 1.0829 | 1.0690 | 1.1893 |
| beagle_108.jpg | 1.2221 | 1.1455 | 1.1343 | 1.1243 | 1.2179 |
| beagle_118.jpg | 1.4236 | 1.0908 | 1.0637 | 1.0329 | 1.2815 |
| beagle_127.jpg | 1.3204 | 1.0585 | 1.0435 | 1.0240 | 1.2463 |
| beagle_137.jpg | 1.4168 | 1.1829 | 1.1282 | 1.0763 | 1.6215 |
| beagle_146.jpg | 1.3055 | 1.0781 | 1.0531 | 1.0339 | 1.3141 |
| beagle_155.jpg | 1.2368 | 1.1002 | 1.0891 | 1.0735 | 1.1749 |
| beagle_165.jpg | 1.1358 | 1.1183 | 1.0972 | 1.0708 | 1.3649 |
| beagle_174.jpg | 1.1336 | 1.0859 | 1.0653 | 1.0435 | 1.3290 |
| beagle_183.jpg | 1.2820 | 1.0580 | 1.0263 | 0.9902 | 1.2572 |
| beagle_192.jpg | 1.2892 | 1.1707 | 1.1598 | 1.1494 | 1.1916 |
| beagle_200.jpg | 1.2399 | 1.1319 | 1.1051 | 1.0824 | 1.3949 |
| beagle_26.jpg | 1.3072 | 1.0749 | 1.0633 | 1.0463 | 1.2482 |
| beagle_35.jpg | 0.9782 | 0.9584 | 0.9451 | 0.9379 | 1.0337 |
| beagle_44.jpg | 0.8730 | 1.0312 | 1.0268 | 1.0286 | 1.0872 |
| beagle_53.jpg | 1.2257 | 1.1653 | 1.1563 | 1.1432 | 1.2626 |
| beagle_62.jpg | 0.9743 | 0.9866 | 0.9790 | 0.9736 | 1.0532 |
| beagle_71.jpg | 1.1250 | 1.0572 | 1.0517 | 1.0455 | 1.1320 |
| beagle_80.jpg | 0.8428 | 0.9076 | 0.9135 | 0.9203 | 0.8789 |
| beagle_9.jpg | 0.9088 | 0.9459 | 0.9147 | 0.9071 | 1.0742 |
| beagle_99.jpg | 0.7444 | 0.9840 | 0.9633 | 0.9623 | 1.0528 |

## Pairwise Win Rate

| Method | IG | Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | NAA |
| --- | ---: | ---: | ---: | ---: | ---: |
| IG | — | 0.7100 | 0.7400 | 0.7600 | 0.1800 |
| Cheap-IG+[0,0.2]/k8000/naa_scaled/rho1 | 0.2900 | — | 0.9700 | 0.9400 | 0.0700 |
| Cheap-IG+[0,0.2]/k16000/naa_scaled/rho1 | 0.2600 | 0.0300 | — | 0.8800 | 0.0400 |
| Cheap-IG+[0,0.2]/k32000/naa_scaled/rho1 | 0.2400 | 0.0600 | 0.1200 | — | 0.0400 |
| NAA | 0.8200 | 0.9300 | 0.9600 | 0.9600 | — |

: 